**Índice**   
1. [Paqueterías necesarias](#1-paqueterías-necesarias)
2. [Filtro de jovenes y adultos mayores](#2-filtro-de-jovenes-y-adultos-mayores)
3. [Configuración](#3-configuración)
4. [Implementación de ViT](#4-implementación-vit)

## 1. Paqueterías necesarias

In [1]:
from FairFaceDatasetZip import FairFaceDatasetZip
from torchvision import transforms
from make_ssampler import make_loader_with_sampler
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os, random
import torch
import torch.nn as nn
import pandas as pd
import pandas as pd
import os
import shutil
import torch as th
import numpy as np
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, WeightedRandomSampler
from pathlib import Path
from tqdm import tqdm
import torch.nn.functional as F
from torchvision import transforms, utils


## 2. Filtro de jovenes y adultos mayores

In [ ]:
df = pd.read_csv('fairface_label_train.csv')
df_young = df[df['age'] == '20-29']
df_young.to_csv('fairface_label_train_young.csv', index=False)



In [ ]:

df = pd.read_csv('fairface_label_train_young.csv')

os.makedirs('young', exist_ok=True)

for filepath in df['file']:
    src = filepath  
    dst = os.path.join('young', os.path.basename(filepath))
    
    if os.path.exists(src):
        shutil.copy(src, dst)



In [ ]:
df = pd.read_csv('fairface_label_train.csv')
df_old = df[df['age'] == 'more than 70']
df_old.to_csv('fairface_label_train_old.csv', index=False)

In [ ]:
df = pd.read_csv('fairface_label_train_old.csv')

os.makedirs('old', exist_ok=True)

for filepath in df['file']:
    src = filepath  
    dst = os.path.join('old', os.path.basename(filepath))
    
    if os.path.exists(src):
        shutil.copy(src, dst)

### Definición de conjunto VAL

In [ ]:
df = pd.read_csv('fairface_label_train.csv')
df_old = df[df['age'] == 'more than 70']
df_old.to_csv('fairface_label_train_old_VAL.csv', index=False)

In [ ]:
df = pd.read_csv('fairface_label_train_old_VAL.csv')

os.makedirs('old', exist_ok=True)

for filepath in df['file']:
    src = filepath  
    dst = os.path.join('old', os.path.basename(filepath))
    
    if os.path.exists(src):
        shutil.copy(src, dst)

## 3. Configuración

In [ ]:
IMG_SIZE = 128
PATCH = 16              
DIM = 256              
DEPTH = 6               # capas transformer
HEADS = 8
BATCH = 32             
EPOCHS = 120
LR = 2e-4
LAMBDA_CYCLE = 10.0
LAMBDA_ID = 5.0
SAVE_EVERY = 5
OUT_DIR = "outputs"
DATA_Y = "young"        
DATA_O = "old"          
# ------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUT_DIR, exist_ok=True)


t = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [ ]:

class FolderDataset(Dataset):
    def __init__(self, folder, transform):
        self.paths = sorted([os.path.join(folder,f) for f in os.listdir(folder) if f.lower().endswith(('.jpg','.png','.jpeg'))])
        self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.transform(img)
    

ds_y = FolderDataset(DATA_Y, t)
ds_o = FolderDataset(DATA_O, t)


In [ ]:
if __name__ == "__main__":
    dl_y = make_loader_with_sampler(ds_y, BATCH)
    dl_o = make_loader_with_sampler(ds_o, BATCH)


## 4. Implementación Vit

In [ ]:
IMG_SIZE = 128
PATCH = 16             
DIM = 256               
DEPTH = 6               # capas transformer
HEADS = 8
BATCH = 64             
EPOCHS = 50
LR = 2e-4
LAMBDA_CYCLE = 10.0
LAMBDA_ID = 5.0
SAVE_EVERY = 5
OUT_DIR = "outputs_vit_cycle"
DATA_Y = "young"        
DATA_O = "old"         
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUT_DIR, exist_ok=True)
t = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])


In [ ]:

class ViTGen(nn.Module):
    def __init__(self, img_size=IMG_SIZE, patch=PATCH, dim=DIM, depth=DEPTH, heads=HEADS):
        super().__init__()
        self.patch = patch
        self.dim = dim
        self.proj = nn.Conv2d(3, dim, kernel_size=patch, stride=patch)  
        n_patches = (img_size // patch) ** 2
        self.pos = nn.Parameter(torch.randn(1, n_patches, dim))
        enc_layer = nn.TransformerEncoderLayer(d_model=dim, nhead=heads, dim_feedforward=dim*4, batch_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=depth)
        self.deproj = nn.Sequential(
            nn.ConvTranspose2d(dim, 256, 4, 2, 1), nn.ReLU(),   # 8 -> 16
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.ReLU(),   # 16 -> 32
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.ReLU(),    # 32 -> 64
            nn.ConvTranspose2d(64, 3, 4, 2, 1), nn.Tanh()       # 64 -> 128
        )

    def forward(self, x):
        B = x.shape[0]
        x = self.proj(x)                     
        C, H, W = x.shape[1], x.shape[2], x.shape[3]
        x = x.flatten(2).transpose(1,2)      
        x = x + self.pos
        x = self.transformer(x)             
        x = x.transpose(1,2).view(B, C, H, W) 
        x = self.deproj(x)
        return x


class Disc128(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1),               
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(64, 128, 4, 2, 1),           
            nn.InstanceNorm2d(128), nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(128, 256, 4, 2, 1),           
            nn.InstanceNorm2d(256), nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(256, 512, 4, 2, 1),           
            nn.InstanceNorm2d(512), nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(512, 1, 4, 1, 0)           
        )

    def forward(self, x):
        return self.net(x)

G_YO = ViTGen().to(device)   
G_OY = ViTGen().to(device)   
D_Y = Disc128().to(device)
D_O = Disc128().to(device)

optG = torch.optim.Adam(list(G_YO.parameters()) + list(G_OY.parameters()), lr=LR, betas=(0.5,0.999))
optD = torch.optim.Adam(list(D_Y.parameters()) + list(D_O.parameters()), lr=LR, betas=(0.5,0.999))

mse = nn.MSELoss()
l1 = nn.L1Loss()


def save_preview(y, o, fake_o, fake_y, epoch, step):
    to_save = torch.cat([y[:4], o[:4], fake_o[:4], fake_y[:4]], dim=0)
    to_save = (to_save + 1) / 2
    to_save = F.interpolate(to_save, size=(256,256), mode='nearest')
    utils.save_image(to_save, os.path.join(OUT_DIR, f"preview_e{epoch}_s{step}.png"), nrow=4)


In [ ]:

global_step = 0

for epoch in range(1, EPOCHS+1):
    pbar = tqdm(zip(dl_y, dl_o), leave=False)

    for y, o in pbar:
        y = y.to(device)
        o = o.to(device)

        with torch.no_grad():
            fake_o = G_YO(y)
            fake_y = G_OY(o)

        D_Y_real = D_Y(y)
        D_Y_fake = D_Y(fake_y.detach())
        D_O_real = D_O(o)
        D_O_fake = D_O(fake_o.detach())

        real_label_Y = torch.ones_like(D_Y_real)
        fake_label_Y = torch.zeros_like(D_Y_fake)
        real_label_O = torch.ones_like(D_O_real)
        fake_label_O = torch.zeros_like(D_O_fake)

        d_loss = (
            mse(D_Y_real, real_label_Y) +
            mse(D_Y_fake, fake_label_Y) +
            mse(D_O_real, real_label_O) +
            mse(D_O_fake, fake_label_O)
        ) * 0.5

        optD.zero_grad()
        d_loss.backward()
        optD.step()
        fake_o = G_YO(y)
        fake_y = G_OY(o)

        D_Y_fake_g = D_Y(fake_y)
        D_O_fake_g = D_O(fake_o)
        real_label_Y = torch.ones_like(D_Y_fake_g)
        real_label_O = torch.ones_like(D_O_fake_g)

        adv_loss = mse(D_Y_fake_g, real_label_Y) + mse(D_O_fake_g, real_label_O)
        cycle_loss = l1(G_OY(fake_o), y) * LAMBDA_CYCLE + l1(G_YO(fake_y), o) * LAMBDA_CYCLE
        id_loss = l1(G_YO(o), o) * LAMBDA_ID + l1(G_OY(y), y) * LAMBDA_ID

        g_loss = adv_loss + cycle_loss + id_loss

        optG.zero_grad()
        g_loss.backward()
        optG.step()

        global_step += 1
        if global_step % 200 == 0:
            save_preview(y, o, fake_o, fake_y, epoch, global_step)

        pbar.set_description(f"E{epoch} D={d_loss.item():.4f} G={g_loss.item():.4f}")


In [ ]:

G_YO = ViTGen().to(device)
ckpt = torch.load("vit_results/vit_results/ckpt_50.pt", map_location=device)
G_YO.load_state_dict(ckpt['G_YO'])
G_YO.eval()

t = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

img = Image.open("TEST.jpg").convert("RGB")
x = t(img).unsqueeze(0).to(device)


with torch.no_grad():
    out = G_YO(x)

out = (out + 1) / 2
out = out.clamp(0,1)

out_img = transforms.ToPILImage()(out.squeeze().cpu())
out_img.save("resultado.png")

print("Listo mi buen")
